# R notebook for regression models and figure export

This notebook is split into separate cells and does the following:

- prepares the 1997–2023 dataset and saves it to CSV
- runs regression equation (5)
- runs regression equation (6) based on principal components
- prepares the standardized datasets used for the coefficient variation diagrams
- saves all figure outputs as **PDF** files using `cairo_pdf(..., fallback_resolution = 300)`
- saves key text outputs to `.txt` files

All generated files are written into these folders relative to the notebook location:

- `data/`
- `outputs/`


In [1]:
# Setup ----------------------------------------------------------------------

dir.create("data", showWarnings = FALSE)
dir.create("outputs", showWarnings = FALSE)

install_if_missing <- function(pkgs) {
  to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(to_install) > 0) {
    install.packages(to_install, repos = "https://cloud.r-project.org")
  }
}

install_if_missing(c("lars", "glmnet"))

library(lars)
library(glmnet)

save_plot_pdf <- function(filename, plot_expr, width = 7, height = 6) {
  grDevices::cairo_pdf(
    filename = file.path("outputs", filename),
    width = width,
    height = height,
    fallback_resolution = 300
  )
  on.exit(dev.off(), add = TRUE)
  eval.parent(substitute(plot_expr))
}

standardize_for_path <- function(v) {
  v <- v - mean(v)
  k <- length(v) / sqrt(drop(crossprod(v)))
  v * k
}

Installing packages into 'C:/Users/it08d/AppData/Local/R/win-library/4.4'
(as 'lib' is unspecified)

also installing the dependencies 'iterators', 'foreach', 'shape', 'RcppEigen'




package 'iterators' successfully unpacked and MD5 sums checked
package 'foreach' successfully unpacked and MD5 sums checked
package 'shape' successfully unpacked and MD5 sums checked
package 'RcppEigen' successfully unpacked and MD5 sums checked
package 'lars' successfully unpacked and MD5 sums checked
package 'glmnet' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\it08d\AppData\Local\Temp\RtmpGWsF5W\downloaded_packages


Loaded lars 1.3


Warning message:
"package 'glmnet' was built under R version 4.4.3"
Loading required package: Matrix

Loaded glmnet 4.1-10



## 1. Regression equation (5): prepare dataset and save CSV

In [2]:
# Regression equation (5) data ------------------------------------------------

year <- 1997:2023

# 1997-2023 price of commercial housing
y <- c(1997,2063,2053,2112,2170,2250,2359,2714,3168,3367,3864,3800,4681,5045,5384,5839,6328,6427,6932,7699,8160,9045,9673,10248,10546,10210,10438) / 1997

# 1997-2023 Chinese gross domestic product
x1 <- c(79715.0,85195.5,90564.4,100280.1,110863.1,121717.4,137422.0,161840.2,187318.9,219438.5,270092.3,319244.6,348517.7,412119.3,487940.2,538580.0,592963.2,643563.1,688858.2,746395.1,832035.9,919281.1,986515.2,1013567.0,1149237.0,1204724.0,1260582.1) / 79715.0

# 1997-2023 Chinese fiscal revenue
x2 <- c(8651.14,9875.95,11444.08,13395.23,16386.04,18903.64,21715.25,26396.47,31649.29,38760.20,51321.78,61330.35,68518.30,83101.51,103874.43,117253.52,129209.64,140370.03,152269.23,159269.97,172592.77,183359.84,190390.08,182913.88,202554.64,203649.29,216795.43) / 8651.14

# 1997-2023 Chinese broad money supply
x3 <- c(90995.3,104498.5,119897.9,134610.3,158301.9,185007.0,221222.8,254107.0,298755.7,345577.9,403442.2,475166.6,610224.5,725851.8,851590.9,974148.8,1106525.0,1228374.8,1392278.1,1550066.7,1690235.3,1826744.2,1986488.8,2186795.9,2382899.6,2664320.8,2922713.3) / 90995.3

# 1997-2023 Chinese consumer price index
x4 <- c(100,99.2,98.6,100.4,100.7,99.2,101.2,103.9,101.8,101.5,104.8,105.9,99.3,103.3,105.4,102.6,102.6,102.0,101.4,102.0,101.6,102.1,102.9,102.5,100.9,102.0,100.2) / 100
for (i in 2:27) {
  x4[i] <- x4[i - 1] * x4[i]
}

# 1997-2023 fixed assets investment of the whole society in China
x5 <- c(24941,28406,29855,32918,37214,43500,53841,66235,80994,97583,118323,144587,181760,218834,205036,241746,282486,320331,347827,372021,394926,418215,439541,451155,473003,495966,509708) / 24941

# 1997-2023 the cost of commercial housing in China
x6 <- c(1175,1218,1152,1139,1128,1184,1273,1402,1451,1564,1657,1795,2021,2228,2373,2498,2643,2816,3054,3039,3105,3210,3549,3781,3891,4093,4150) / 1175

# 1997-2023 per capita disposable income of urban residents in China
x7 <- c(5160.3,5425.1,5854.0,6280.0,6859.6,7702.8,8472.2,9421.6,10493.0,11759.5,13785.8,15780.8,17174.7,19109.4,21809.8,24564.7,26955.1,28843.9,31194.8,33616.2,36396.2,39250.8,42358.8,43833.8,47412,49283,51821) / 5160.3

X <- data.frame(year, x1, x2, x3, x4, x5, x6, x7, y)

write.csv(X, file = file.path("data", "regression_data_1997_2023.csv"), row.names = FALSE)

X

year,x1,x2,x3,x4,x5,x6,x7,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1997,1.000000,1.000000,1.000000,1.0000000,1.000000,1.0000000,1.000000,1.000000
1998,1.068751,1.141578,1.148394,0.9920000,1.138928,1.0365957,1.051315,1.033050
1999,1.136102,1.322841,1.317627,0.9781120,1.197025,0.9804255,1.134430,1.028042
2000,1.257983,1.548377,1.479310,0.9820244,1.319835,0.9693617,1.216984,1.057586
2001,1.390743,1.894090,1.739671,0.9888986,1.492081,0.9600000,1.329303,1.086630
2002,1.526907,2.185104,2.033149,0.9809874,1.744116,1.0076596,1.492704,1.126690
2003,1.723916,2.510103,2.431145,0.9927593,2.158735,1.0834043,1.641804,1.181272
2004,2.030235,3.051213,2.792529,1.0314769,2.655667,1.1931915,1.825785,1.359039
2005,2.349858,3.658395,3.283199,1.0500435,3.247424,1.2348936,2.033409,1.586380


## 2. Fit regression equation (5) and save summary

In [3]:
# Regression equation (5) -----------------------------------------------------

hp <- lm(y ~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X)

summary_hp <- summary(hp)
summary_hp

capture.output(summary_hp, file = file.path("outputs", "regression_equation_5_summary.txt"))


Call:
lm(formula = y ~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.26512 -0.05632  0.01297  0.05291  0.23561 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)   
(Intercept)  0.64371    1.53949   0.418  0.68054   
x1           0.11011    0.18589   0.592  0.56062   
x2          -0.08653    0.04368  -1.981  0.06223 . 
x3          -0.13567    0.04616  -2.939  0.00842 **
x4          -0.78316    1.94359  -0.403  0.69149   
x5           0.07412    0.06338   1.169  0.25668   
x6           0.23589    0.49223   0.479  0.63725   
x7           0.83106    0.33954   2.448  0.02427 * 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.1206 on 19 degrees of freedom
Multiple R-squared:  0.9955,	Adjusted R-squared:  0.9938 
F-statistic: 596.3 on 7 and 19 DF,  p-value: < 2.2e-16


## 3. Fit regression equation (6) via principal components and save summary

In [4]:
# Regression equation (6) -----------------------------------------------------

hp.pr <- princomp(~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X, cor = TRUE)
summary_hp_pr <- summary(hp.pr, loadings = TRUE)
summary_hp_pr

pre <- predict(hp.pr)
X$z1 <- pre[, 1]

lm.sol <- lm(y ~ z1, data = X)
summary_lm_sol <- summary(lm.sol)
summary_lm_sol

beta <- coef(lm.sol)
A <- loadings(hp.pr)
x.bar <- hp.pr$center
x.sd <- hp.pr$scale
coef_pc <- beta[2] * A[, 1] / x.sd
beta0 <- beta[1] - sum(x.bar * coef_pc)

pc_regression_coefficients <- c(beta0 = beta0, coef_pc)
pc_regression_coefficients

capture.output(summary_hp_pr, file = file.path("outputs", "regression_equation_6_pca_summary.txt"))
capture.output(summary_lm_sol, file = file.path("outputs", "regression_equation_6_lm_summary.txt"))
write.csv(
  data.frame(term = names(pc_regression_coefficients), value = as.numeric(pc_regression_coefficients)),
  file = file.path("outputs", "regression_equation_6_coefficients.csv"),
  row.names = FALSE
)

Importance of components:
                          Comp.1      Comp.2      Comp.3       Comp.4
Standard deviation     2.6349591 0.206567137 0.085060511 0.0620737031
Proportion of Variance 0.9918585 0.006095712 0.001033613 0.0005504492
Cumulative Proportion  0.9918585 0.997954172 0.998987785 0.9995382345
                             Comp.5       Comp.6       Comp.7
Standard deviation     0.0428412881 0.0326148374 0.0182552614
Proportion of Variance 0.0002621966 0.0001519611 0.0000476078
Cumulative Proportion  0.9998004311 0.9999523922 1.0000000000

Loadings:
   Comp.1 Comp.2 Comp.3 Comp.4 Comp.5 Comp.6 Comp.7
x1  0.379  0.309         0.339  0.293  0.370  0.647
x2  0.377 -0.435  0.512  0.251 -0.578              
x3  0.376  0.674               -0.328 -0.536       
x4  0.378 -0.414 -0.464  0.409  0.245 -0.493       
x5  0.378 -0.249  0.315 -0.697  0.381 -0.193  0.166
x6  0.379        -0.629 -0.382 -0.374  0.417       
x7  0.379  0.162  0.144  0.138  0.357  0.336 -0.741


Call:
lm(formula = y ~ z1, data = X)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.29860 -0.05156 -0.01894  0.10229  0.34340 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  2.75547    0.02894   95.21   <2e-16 ***
z1           0.56788    0.01098   51.70   <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.1504 on 25 degrees of freedom
Multiple R-squared:  0.9907,	Adjusted R-squared:  0.9904 
F-statistic:  2673 on 1 and 25 DF,  p-value: < 2.2e-16


beta0.(Intercept)                x1                x2                x3 
      -0.38999198        0.04498709        0.02562115        0.02259855 
               x4                x5                x6                x7 
       0.97450980        0.03190381        0.25029377        0.07354322

## 4. Prepare standardized data for coefficient variation diagrams (Figures 1–3)

In [5]:
# Standardized data for coefficient variation diagrams 1, 2, 3 ---------------

y_std <- c(1997,2063,2053,2112,2170,2250,2359,2714,3168,3367,3864,3800,4681,5045,5384,5839,6328,6427,6932,7699,8160,9045,9673,10248,10546,10210,10438) / 1997
y_std <- y_std - mean(y_std)

x1_std <- c(79715.0,85195.5,90564.4,100280.1,110863.1,121717.4,137422.0,161840.2,187318.9,219438.5,270092.3,319244.6,348517.7,412119.3,487940.2,538580.0,592963.2,643563.1,688858.2,746395.1,832035.9,919281.1,986515.2,1013567.0,1149237.0,1204724.0,1260582.1) / 79715.0
x1_std <- standardize_for_path(x1_std)

x2_std <- c(8651.14,9875.95,11444.08,13395.23,16386.04,18903.64,21715.25,26396.47,31649.29,38760.20,51321.78,61330.35,68518.30,83101.51,103874.43,117253.52,129209.64,140370.03,152269.23,159269.97,172592.77,183359.84,190390.08,182913.88,202554.64,203649.29,216795.43) / 8651.14
x2_std <- standardize_for_path(x2_std)

x3_std <- c(90995.3,104498.5,119897.9,134610.3,158301.9,185007.0,221222.8,254107.0,298755.7,345577.9,403442.2,475166.6,610224.5,725851.8,851590.9,974148.8,1106525.0,1228374.8,1392278.1,1550066.7,1690235.3,1826744.2,1986488.8,2186795.9,2382899.6,2664320.8,2922713.3) / 90995.3
x3_std <- standardize_for_path(x3_std)

x4_std <- c(100,99.2,98.6,100.4,100.7,99.2,101.2,103.9,101.8,101.5,104.8,105.9,99.3,103.3,105.4,102.6,102.6,102.0,101.4,102.0,101.6,102.1,102.9,102.5,100.9,102.0,100.2) / 100
for (i in 2:27) {
  x4_std[i] <- x4_std[i - 1] * x4_std[i]
}
x4_std <- standardize_for_path(x4_std)

x5_std <- c(24941,28406,29855,32918,37214,43500,53841,66235,80994,97583,118323,144587,181760,218834,205036,241746,282486,320331,347827,372021,394926,418215,439541,451155,473003,495966,509708) / 24941
x5_std <- standardize_for_path(x5_std)

x6_std <- c(1175,1218,1152,1139,1128,1184,1273,1402,1451,1564,1657,1795,2021,2228,2373,2498,2643,2816,3054,3039,3105,3210,3549,3781,3891,4093,4150) / 1175
x6_std <- standardize_for_path(x6_std)

x7_std <- c(5160.3,5425.1,5854.0,6280.0,6859.6,7702.8,8472.2,9421.6,10493.0,11759.5,13785.8,15780.8,17174.7,19109.4,21809.8,24564.7,26955.1,28843.9,31194.8,33616.2,36396.2,39250.8,42358.8,43833.8,47412,49283,51821) / 5160.3
x7_std <- standardize_for_path(x7_std)

X_std <- cbind(x1_std, x2_std, x3_std, x4_std, x5_std, x6_std, x7_std)

standardized_data <- data.frame(
  year = 1997:2023,
  x1 = x1_std, x2 = x2_std, x3 = x3_std, x4 = x4_std,
  x5 = x5_std, x6 = x6_std, x7 = x7_std, y = y_std
)

write.csv(
  standardized_data,
  file = file.path("data", "standardized_data_1997_2023.csv"),
  row.names = FALSE
)

head(standardized_data)

,year,x1,x2,x3,x4,x5,x6,x7,y
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1997,-5.838348,-6.334849,-5.473377,-5.796556,-6.228377,-5.882467,-6.130846,-1.755467
2,1998,-5.763590,-6.246915,-5.391707,-5.985456,-6.121216,-5.661189,-6.039772,-1.722417
3,1999,-5.690355,-6.134332,-5.298569,-6.313385,-6.076403,-6.000826,-5.892258,-1.727424
4,2000,-5.557826,-5.994250,-5.209585,-6.221003,-5.981674,-6.067724,-5.745742,-1.697880
5,2001,-5.413467,-5.779527,-5.066294,-6.058687,-5.848812,-6.124330,-5.546397,-1.668837
6,2002,-5.265407,-5.598777,-4.904777,-6.245490,-5.654406,-5.836153,-5.256391,-1.628776


## 5. Generate and save Figures 1–3 as PDF

In [6]:
# Figures 1, 2, 3 ------------------------------------------------------------

c_lasso_1 <- lars(X_std, y_std, type = "lasso")
a_lar_1 <- lars(X_std, y_std, type = "lar")
b_glmnet_1 <- glmnet(X_std, y_std, alpha = 1, family = "gaussian")

save_plot_pdf("figure1_lasso_path_1997_2023.pdf", plot(c_lasso_1))
save_plot_pdf("figure2_lar_path_1997_2023.pdf", plot(a_lar_1))
save_plot_pdf("figure3_glmnet_path_1997_2023.pdf", plot(b_glmnet_1))

lar_beta_1 <- lars(X_std, y_std, type = "lar")$beta
lasso_beta_1 <- lars(X_std, y_std, type = "lasso")$beta
glmnet_beta_1 <- as.matrix(glmnet(X_std, y_std, alpha = 1, family = "gaussian")$beta)

lar_beta_1
lasso_beta_1
glmnet_beta_1

write.csv(lar_beta_1, file = file.path("outputs", "figure1_3_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_1, file = file.path("outputs", "figure1_3_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_1, file = file.path("outputs", "figure1_3_glmnet_beta.csv"), row.names = TRUE)

,x1_std,x2_std,x3_std,x4_std,x5_std,x6_std,x7_std
0,0.000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000
1,0.000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.09863443
2,0.000000,0.00000000,0.00000000,0.000000000,0.05725972,0.00000000,0.15589415
3,0.000000,0.00000000,0.00000000,0.032754300,0.06673289,0.00000000,0.18879449
4,0.000000,0.00000000,-0.05080159,0.009595910,0.04943748,0.00000000,0.27954915
5,0.000000,0.00000000,-0.07630591,-0.008246843,0.03994435,0.01455839,0.31756072
6,0.000000,-0.06060263,-0.14021239,-0.015086558,0.05682650,0.02396027,0.42182469
7,0.101259,-0.13931289,-0.24651985,-0.033167085,0.09609610,0.03901240,0.46825608


,x1_std,x2_std,x3_std,x4_std,x5_std,x6_std,x7_std
0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000
1,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000000,0.09863443
2,0.00000000,0.00000000,0.00000000,0.00000000,0.05725972,0.000000000,0.15589415
3,0.00000000,0.00000000,0.00000000,0.03275430,0.06673289,0.000000000,0.18879449
4,0.00000000,0.00000000,-0.05080159,0.00959591,0.04943748,0.000000000,0.27954915
5,0.00000000,0.00000000,-0.06451792,0.00000000,0.04433203,0.007829564,0.29999194
6,0.00000000,0.00000000,-0.06928378,0.00000000,0.04064973,0.008314629,0.30792361
7,0.00000000,-0.07152128,-0.13783090,0.00000000,0.06126393,0.013299038,0.42153979
8,0.05933819,-0.12531396,-0.19845498,0.00000000,0.08739248,0.014632282,0.44854869
9,0.10125900,-0.13931289,-0.24651985,-0.03316708,0.09609610,0.039012401,0.46825608


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s45,s46,s47,s48,s49,s50,s51,s52,s53,s54
x1_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x2_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x3_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x4_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.02270462,0.02285529,0.02300565,0.02320507,0.02335521,0.02350544,0.02365572,0.02375635,0.02385708,0.02395792
x5_std,0,0.00000000,0.00000000,0.00000000,0.003777238,0.01262400,0.02072126,0.02804779,0.03471738,0.04080972,⋯,0.07728137,0.07735347,0.07739074,0.07730641,0.07728727,0.07724358,0.07717816,0.07718226,0.07716711,0.07713477
x6_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x7_std,0,0.02559004,0.04890673,0.07015203,0.085752905,0.09459165,0.10260895,0.10996514,0.11667386,0.12277143,⋯,0.18430913,0.18447705,0.18464537,0.18485453,0.18501901,0.18518169,0.18534210,0.18546087,0.18557891,0.18569593


## 6. Prepare data for Figures 4–6 and save CSV

In [7]:
# Figures 4, 5, 6 data --------------------------------------------------------

x1_f46 <- c(10,20,30,0,0,0,0,0,0,0,0,0,0,0,0,0)
x1_f46 <- standardize_for_path(x1_f46)

x2_f46 <- c(0,0,0,40,50,60,70,80,9,10,0,0,0,0,0,0)
x2_f46 <- standardize_for_path(x2_f46)

x3_f46 <- c(0,0,0,0,0,0,0,0,0,0,11,12,13,0,0,0)
x3_f46 <- standardize_for_path(x3_f46)

x4_f46 <- c(0,0,0,0,0,0,0,0,0,0,0,0,0,14,15,16)
x4_f46 <- standardize_for_path(x4_f46)

X_f46 <- cbind(x1_f46, x2_f46, x3_f46, x4_f46)
y_f46 <- c(-1,10,-5,-10,1,8,3,4,2,1,9,7,5,3,2,-1)
y_f46 <- y_f46 - mean(y_f46)

figure46_data <- data.frame(
  obs = 1:16,
  x1 = x1_f46, x2 = x2_f46, x3 = x3_f46, x4 = x4_f46, y = y_f46
)

write.csv(
  figure46_data,
  file = file.path("data", "figure4_6_data.csv"),
  row.names = FALSE
)

figure46_data

obs,x1,x2,x3,x4,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2.91730,-2.817285,-1.916087,-1.918044,-3.375
2,7.58498,-2.817285,-1.916087,-1.918044,7.625
3,12.25266,-2.817285,-1.916087,-1.918044,-7.375
4,-1.75038,2.834948,-1.916087,-1.918044,-12.375
5,-1.75038,4.248007,-1.916087,-1.918044,-1.375
6,-1.75038,5.661065,-1.916087,-1.918044,5.625
7,-1.75038,7.074123,-1.916087,-1.918044,0.625
8,-1.75038,8.487182,-1.916087,-1.918044,1.625
9,-1.75038,-1.545533,-1.916087,-1.918044,-0.375


## 7. Generate and save Figures 4–6 as PDF

In [8]:
# Figures 4, 5, 6 ------------------------------------------------------------

c_lasso_46 <- lars(X_f46, y_f46, type = "lasso")
a_lar_46 <- lars(X_f46, y_f46, type = "lar")
b_glmnet_46 <- glmnet(X_f46, y_f46, alpha = 1, family = "gaussian")

save_plot_pdf("figure4_lasso_path_block.pdf", plot(c_lasso_46))
save_plot_pdf("figure5_lar_path_block.pdf", plot(a_lar_46))
save_plot_pdf("figure6_glmnet_path_block.pdf", plot(b_glmnet_46))

lar_beta_46 <- lars(X_f46, y_f46, type = "lar")$beta
lasso_beta_46 <- lars(X_f46, y_f46, type = "lasso")$beta
glmnet_beta_46 <- as.matrix(glmnet(X_f46, y_f46, alpha = 1, family = "gaussian")$beta)

lar_beta_46
lasso_beta_46
glmnet_beta_46

write.csv(lar_beta_46, file = file.path("outputs", "figure4_6_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_46, file = file.path("outputs", "figure4_6_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_46, file = file.path("outputs", "figure4_6_glmnet_beta.csv"), row.names = TRUE)

,x1_f46,x2_f46,x3_f46,x4_f46
0,0.000000000,0.00000000,0.0000000,0.00000000
1,0.000000000,0.00000000,0.4329299,0.00000000
2,0.000000000,0.07937655,0.5123064,0.00000000
3,-0.007303067,0.11630382,0.5497721,0.00000000
4,0.037686411,0.22368302,0.6460540,0.09631987


,x1_f46,x2_f46,x3_f46,x4_f46
0,0.000000000,0.00000000,0.0000000,0.00000000
1,0.000000000,0.00000000,0.4329299,0.00000000
2,0.000000000,0.07937655,0.5123064,0.00000000
3,-0.007303067,0.11630382,0.5497721,0.00000000
4,0.000000000,0.13373451,0.5654013,0.01563544
5,0.000000000,0.17828899,0.6066248,0.05687138
6,0.037686411,0.22368302,0.6460540,0.09631987


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s62,s63,s64,s65,s66,s67,s68,s69,s70,s71
x1_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.02827311,0.02904884,0.02979361,0.03048638,0.03092923,0.03163398,0.03203542,0.03246456,0.03288872,0.03329325
x2_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.21236521,0.21330012,0.21419622,0.21502920,0.21557767,0.21641122,0.21690640,0.21742451,0.21793464,0.21842028
x3_f46,0,0.0480221,0.09177804,0.1316468,0.1679738,0.2010735,0.2312328,0.2587128,0.2837516,0.306566,⋯,0.63630563,0.63712015,0.63789462,0.63861232,0.63910888,0.63981211,0.64025330,0.64070703,0.64114843,0.64156588
x4_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.08659350,0.08740921,0.08818279,0.08889895,0.08940471,0.09009909,0.09054618,0.09100132,0.09144226,0.09185835


## 8. Prepare data for Figures 7–9 and save CSV

In [9]:
# Figures 7, 8, 9 data --------------------------------------------------------

x1_f79  <- standardize_for_path(c(10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0))
x2_f79  <- standardize_for_path(c(0,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0))
x3_f79  <- standardize_for_path(c(0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0))
x4_f79  <- standardize_for_path(c(0,0,0,40,0,0,0,0,0,0,0,0,0,0,0,0))
x5_f79  <- standardize_for_path(c(0,0,0,0,50,0,0,0,0,0,0,0,0,0,0,0))
x6_f79  <- standardize_for_path(c(0,0,0,0,0,60,0,0,0,0,0,0,0,0,0,0))
x7_f79  <- standardize_for_path(c(0,0,0,0,0,0,70,0,0,0,0,0,0,0,0,0))
x8_f79  <- standardize_for_path(c(0,0,0,0,0,0,0,80,0,0,0,0,0,0,0,0))
x9_f79  <- standardize_for_path(c(0,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0))
x10_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,10,0,0,0,0,0,0))
x11_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0))
x12_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,12,0,0,0,0))
x13_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,13,0,0,0))
x14_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,0))
x15_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0))
x16_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16))

X_f79 <- cbind(
  x1_f79, x2_f79, x3_f79, x4_f79, x5_f79, x6_f79, x7_f79, x8_f79,
  x9_f79, x10_f79, x11_f79, x12_f79, x13_f79, x14_f79, x15_f79, x16_f79
)

y_f79 <- c(-1,10,-5,-10,1,8,3,4,2,1,9,7,5,3,2,-1)
y_f79 <- y_f79 - mean(y_f79)

figure79_data <- data.frame(
  obs = 1:16,
  x1 = x1_f79, x2 = x2_f79, x3 = x3_f79, x4 = x4_f79,
  x5 = x5_f79, x6 = x6_f79, x7 = x7_f79, x8 = x8_f79,
  x9 = x9_f79, x10 = x10_f79, x11 = x11_f79, x12 = x12_f79,
  x13 = x13_f79, x14 = x14_f79, x15 = x15_f79, x16 = x16_f79,
  y = y_f79
)

write.csv(
  figure79_data,
  file = file.path("data", "figure7_9_data.csv"),
  row.names = FALSE
)

figure79_data

obs,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-3.375
2,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,7.625
3,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-7.375
4,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-12.375
5,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.375
6,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,5.625
7,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,0.625
8,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,1.625
9,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-0.375


## 9. Generate and save Figures 7–9 as PDF

In [10]:
# Figures 7, 8, 9 ------------------------------------------------------------

c_lasso_79 <- lars(X_f79, y_f79, type = "lasso")
a_lar_79 <- lars(X_f79, y_f79, type = "lar")
b_glmnet_79 <- glmnet(X_f79, y_f79, alpha = 1, family = "gaussian")

save_plot_pdf("figure7_lasso_path_singletons.pdf", plot(c_lasso_79))
save_plot_pdf("figure8_lar_path_singletons.pdf", plot(a_lar_79))
save_plot_pdf("figure9_glmnet_path_singletons.pdf", plot(b_glmnet_79))

lar_beta_79 <- lars(X_f79, y_f79, type = "lar")$beta
lasso_beta_79 <- lars(X_f79, y_f79, type = "lasso")$beta
glmnet_beta_79 <- as.matrix(glmnet(X_f79, y_f79, alpha = 1, family = "gaussian")$beta)

lar_beta_79
lasso_beta_79
glmnet_beta_79

write.csv(lar_beta_79, file = file.path("outputs", "figure7_9_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_79, file = file.path("outputs", "figure7_9_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_79, file = file.path("outputs", "figure7_9_glmnet_beta.csv"), row.names = TRUE)

,x1_f79,x2_f79,x3_f79,x4_f79,x5_f79,x6_f79,x7_f79,x8_f79,x9_f79,x10_f79,x11_f79,x12_f79,x13_f79,x14_f79,x15_f79,x16_f79
0,0.00000000,0.00000000,0.00000000,0.0000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
1,0.00000000,0.00000000,0.00000000,-0.3025768,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
2,0.00000000,0.00000000,-0.03025768,-0.3328345,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
3,0.00000000,0.06051536,-0.10085894,-0.4034358,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
4,0.00000000,0.12103073,-0.16137431,-0.4639511,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.06051536,0.00000000,0.00000000,0.00000000,0,0.00000000
5,0.00000000,0.18154609,-0.21180378,-0.5143806,0.00000000,0.06051536,0.00000000,0.00000000,0.000000e+00,0.00000000,0.12103073,0.00000000,0.00000000,0.00000000,0,0.00000000
6,0.00000000,0.22693262,-0.24206146,-0.5446383,0.00000000,0.10590189,0.00000000,0.00000000,0.000000e+00,0.00000000,0.16641725,0.04538652,0.00000000,0.00000000,0,0.00000000
7,-0.07564421,0.30257682,-0.31770567,-0.6202825,0.00000000,0.18154609,0.00000000,0.00000000,0.000000e+00,0.00000000,0.24206146,0.12103073,0.00000000,0.00000000,0,-0.07564421
8,-0.12103073,0.36309219,-0.36309219,-0.6656690,0.00000000,0.24206146,0.00000000,0.00000000,0.000000e+00,0.00000000,0.30257682,0.18154609,0.06051536,0.00000000,0,-0.12103073
9,-0.18154609,0.42360755,-0.42360755,-0.7261844,-0.06051536,0.30257682,0.00000000,0.06051536,0.000000e+00,-0.06051536,0.36309219,0.24206146,0.12103073,0.00000000,0,-0.18154609


,x1_f79,x2_f79,x3_f79,x4_f79,x5_f79,x6_f79,x7_f79,x8_f79,x9_f79,x10_f79,x11_f79,x12_f79,x13_f79,x14_f79,x15_f79,x16_f79
0,0.00000000,0.00000000,0.00000000,0.0000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
1,0.00000000,0.00000000,0.00000000,-0.3025768,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
2,0.00000000,0.00000000,-0.03025768,-0.3328345,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
3,0.00000000,0.06051536,-0.10085894,-0.4034358,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
4,0.00000000,0.12103073,-0.16137431,-0.4639511,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.06051536,0.00000000,0.00000000,0.00000000,0,0.00000000
5,0.00000000,0.18154609,-0.21180378,-0.5143806,0.00000000,0.06051536,0.00000000,0.00000000,0.000000e+00,0.00000000,0.12103073,0.00000000,0.00000000,0.00000000,0,0.00000000
6,0.00000000,0.22693262,-0.24206146,-0.5446383,0.00000000,0.10590189,0.00000000,0.00000000,0.000000e+00,0.00000000,0.16641725,0.04538652,0.00000000,0.00000000,0,0.00000000
7,-0.07564421,0.30257682,-0.31770567,-0.6202825,0.00000000,0.18154609,0.00000000,0.00000000,0.000000e+00,0.00000000,0.24206146,0.12103073,0.00000000,0.00000000,0,-0.07564421
8,-0.12103073,0.36309219,-0.36309219,-0.6656690,0.00000000,0.24206146,0.00000000,0.00000000,0.000000e+00,0.00000000,0.30257682,0.18154609,0.06051536,0.00000000,0,-0.12103073
9,-0.18154609,0.42360755,-0.42360755,-0.7261844,-0.06051536,0.30257682,0.00000000,0.06051536,0.000000e+00,-0.06051536,0.36309219,0.24206146,0.12103073,0.00000000,0,-0.18154609


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s38,s39,s40,s41,s42,s43,s44,s45,s46,s47
x1_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.189523031,-0.191365207,-0.19304373,-0.19457313,-0.19596667,-0.19723641,-0.19839335,-0.19944751,-0.20040802,-0.20128321
x2_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.009416637,0.04455803,0.07791482,0.10952073,⋯,0.432486580,0.434522985,0.43637848,0.43806914,0.43960961,0.44101322,0.44229215,0.44345745,0.44451924,0.44548669
x3_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,-0.041243202,-0.08224161,-0.11826045,-0.14986711,⋯,-0.431583586,-0.433425842,-0.43510444,-0.43663391,-0.43802751,-0.43929730,-0.44045429,-0.44150850,-0.44246905,-0.44334427
x4_f79,0,-0.07096344,-0.1356227,-0.1945378,-0.248219,-0.2971314,-0.343820247,-0.38481860,-0.42083652,-0.45244290,⋯,-0.734158973,-0.736001356,-0.73768007,-0.73920965,-0.74060334,-0.74187323,-0.74303030,-0.74408458,-0.74504520,-0.74592048
x5_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.068490109,-0.070332479,-0.07201118,-0.07354075,-0.07493443,-0.07620431,-0.07736137,-0.07841564,-0.07937625,-0.08025153
x6_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.311456567,0.313492909,0.31534835,0.31703895,0.31857937,0.31998294,0.32126183,0.32242710,0.32348885,0.32445627
x7_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.008879842,0.010916175,0.01277161,0.01446221,0.01600262,0.01740618,0.01868506,0.01985032,0.02091207,0.02187949
x8_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.069395285,0.071431611,0.07328704,0.07497763,0.07651804,0.07792160,0.07920047,0.08036573,0.08142747,0.08239489
x9_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.007975203,-0.009817532,-0.01149619,-0.01302573,-0.01441938,-0.01568923,-0.01684627,-0.01790051,-0.01886111,-0.01973636
x10_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.068490790,-0.070333099,-0.07201174,-0.07354126,-0.07493490,-0.07620473,-0.07736176,-0.07841599,-0.07937658,-0.08025182


## 10. Files created by this notebook

After you run all cells, you should get:

### In `data/`
- `regression_data_1997_2023.csv`
- `standardized_data_1997_2023.csv`
- `figure4_6_data.csv`
- `figure7_9_data.csv`

### In `outputs/`
- `regression_equation_5_summary.txt`
- `regression_equation_6_pca_summary.txt`
- `regression_equation_6_lm_summary.txt`
- `regression_equation_6_coefficients.csv`
- `figure1_lasso_path_1997_2023.pdf`
- `figure2_lar_path_1997_2023.pdf`
- `figure3_glmnet_path_1997_2023.pdf`
- `figure4_lasso_path_block.pdf`
- `figure5_lar_path_block.pdf`
- `figure6_glmnet_path_block.pdf`
- `figure7_lasso_path_singletons.pdf`
- `figure8_lar_path_singletons.pdf`
- `figure9_glmnet_path_singletons.pdf`
- beta-matrix CSV files for each figure group
